# Netflix Content Analysis

**Pranay Beemanapalli**  
MS Data Science | Fall 2024  

---

This notebook explores the Netflix titles dataset (~8,800 titles) to answer three questions:

1. What type of content does Netflix have — movies or TV shows?
2. Which countries produce the most content?
3. What genres dominate the platform?

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# global plot settings applied once so all charts stay consistent
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


## 1. Load & Inspect the Data

In [ ]:
df = pd.read_csv('../data/netflix_titles.csv')

print(f"Shape: {df.shape}")
df.head(3)


In [ ]:
# check for missing values before any analysis
df.isnull().sum()


In [ ]:
# extract year from date_added column (format: 'January 1, 2021')
df['year_added'] = df['date_added'].str.extract(r'(\d{4})').astype(float)

df['year_added'].value_counts().sort_index()


## 2. Content Type — Movies vs TV Shows

In [ ]:
counts = df['type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# pie chart for proportion
axes[0].pie(
    counts,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=['#e50914', '#564d8a'],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Content Type Split', fontsize=13, fontweight='bold')

# bar chart for exact counts
bars = axes[1].bar(counts.index, counts.values, color=['#e50914', '#564d8a'], width=0.5)
axes[1].set_title('Count by Type', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Titles')
axes[1].yaxis.grid(True, alpha=0.3)

# add count labels on top of each bar
for bar in bars:
    axes[1].annotate(
        f'{bar.get_height():,}',
        xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
        xytext=(0, 5), textcoords='offset points',
        ha='center', fontsize=12, fontweight='bold'
    )

plt.suptitle('Netflix Library: Movies vs TV Shows', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../dashboard/type_split.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# print summary percentages
print(f"Movies  : {counts['Movie']:,}  ({counts['Movie']/len(df)*100:.1f}%)")
print(f"TV Shows: {counts['TV Show']:,}  ({counts['TV Show']/len(df)*100:.1f}%)")
print(f"Total   : {len(df):,} titles")


### Content Added Per Year

In [ ]:
# group by year and content type to see yearly addition trend
yearly = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
yearly = yearly[yearly.index >= 2015]  # pre-2015 data is sparse

fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(yearly))
w = 0.38

ax.bar(x - w/2, yearly['Movie'],   width=w, label='Movies',   color='#e50914', alpha=0.88)
ax.bar(x + w/2, yearly['TV Show'], width=w, label='TV Shows', color='#564d8a', alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels([int(y) for y in yearly.index])
ax.set_xlabel('Year Added to Netflix')
ax.set_ylabel('Titles Added')
ax.set_title('Content Added to Netflix Per Year (2015–2023)', fontsize=13, fontweight='bold')
ax.yaxis.grid(True, alpha=0.3)
ax.legend()

# mark the peak year
peak_idx = yearly['Movie'].argmax()
ax.annotate('Peak (2021)', xy=(peak_idx - w/2, yearly['Movie'].iloc[peak_idx]),
            xytext=(peak_idx - 1.8, yearly['Movie'].iloc[peak_idx] + 60),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10, color='gray')

plt.tight_layout()
plt.savefig('../dashboard/yearly_additions.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Country Analysis

In [ ]:
# some titles list multiple countries (e.g. 'United States, United Kingdom')
# extracting only the first country for simplicity
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()

top_countries = df['primary_country'].value_counts().dropna().head(15)

fig, ax = plt.subplots(figsize=(11, 7))

# highlight top country in red, rest in gray
colors = ['#e50914' if i == 0 else '#cccccc' for i in range(len(top_countries))]
bars = ax.barh(top_countries.index[::-1], top_countries.values[::-1], color=colors[::-1])

# add count labels at end of each bar
for bar in bars:
    ax.text(bar.get_width() + 15, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', color='#555', fontsize=10)

ax.set_xlabel('Number of Titles')
ax.set_title('Top 15 Countries by Netflix Content Volume', fontsize=13, fontweight='bold')
ax.xaxis.grid(True, alpha=0.3)
ax.set_xlim(0, top_countries.max() * 1.15)

plt.tight_layout()
plt.savefig('../dashboard/top_countries.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
us_share = top_countries['United States'] / df['primary_country'].notna().sum() * 100
print(f"US share of total content: {us_share:.1f}%")
print(f"
Top 5 countries:")
print(top_countries.head(5).to_string())


In [ ]:
# compare movies vs TV shows ratio for top 8 countries
top8 = top_countries.head(8).index.tolist()
country_type = (df[df['primary_country'].isin(top8)]
                .groupby(['primary_country', 'type'])
                .size()
                .unstack(fill_value=0))
country_type['total'] = country_type.sum(axis=1)
country_type = country_type.sort_values('total', ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))

# stacked horizontal bar chart
ax.barh(country_type.index, country_type['Movie'],   color='#e50914', label='Movies',   alpha=0.85)
ax.barh(country_type.index, country_type['TV Show'],
        left=country_type['Movie'], color='#564d8a', label='TV Shows', alpha=0.85)

ax.set_xlabel('Number of Titles')
ax.set_title('Movies vs TV Shows by Country (Top 8)', fontsize=13, fontweight='bold')
ax.xaxis.grid(True, alpha=0.3)
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('../dashboard/country_type_mix.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# calculate TV show percentage per country to compare content mix
for country in ['South Korea', 'Japan', 'India', 'United States']:
    sub = df[df['primary_country'] == country]['type'].value_counts()
    if 'TV Show' in sub and 'Movie' in sub:
        tv_pct = sub['TV Show'] / sub.sum() * 100
        print(f"{country:<15} — TV Show share: {tv_pct:.1f}%")


## 4. Genre Analysis

In [ ]:
# split movies and TV shows for separate genre analysis
movies = df[df['type'] == 'Movie'].copy()
shows  = df[df['type'] == 'TV Show'].copy()

# note: listed_in can contain multiple genres — using first genre only
top_movie_genres = movies['listed_in'].value_counts().head(10)
top_tv_genres    = shows['listed_in'].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# movie genres
axes[0].barh(top_movie_genres.index[::-1], top_movie_genres.values[::-1],
             color='#e50914', alpha=0.85)
axes[0].set_title('Top Movie Genres', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].xaxis.grid(True, alpha=0.3)
for i, val in enumerate(top_movie_genres.values[::-1]):
    axes[0].text(val + 10, i, str(val), va='center', fontsize=9, color='#555')

# tv show genres
axes[1].barh(top_tv_genres.index[::-1], top_tv_genres.values[::-1],
             color='#564d8a', alpha=0.85)
axes[1].set_title('Top TV Show Genres', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Count')
axes[1].xaxis.grid(True, alpha=0.3)
for i, val in enumerate(top_tv_genres.values[::-1]):
    axes[1].text(val + 5, i, str(val), va='center', fontsize=9, color='#555')

plt.suptitle('Genre Distribution — Movies vs TV Shows', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../dashboard/genres.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Ratings Distribution

In [ ]:
rating_counts = df['rating'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))

# highlight adult ratings in red, others in gray
bar_colors = ['#e50914' if r in ['TV-MA', 'R'] else '#aaaaaa' for r in rating_counts.index]
bars = ax.bar(rating_counts.index, rating_counts.values, color=bar_colors)

ax.set_title('Audience Rating Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Rating')
ax.set_ylabel('Number of Titles')
ax.yaxis.grid(True, alpha=0.3)

# label bars with count where value is large enough to matter
for bar in bars:
    if bar.get_height() > 200:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{int(bar.get_height()):,}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('../dashboard/ratings.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# summarize adult vs family-friendly content
adult = rating_counts[['TV-MA', 'R']].sum()
kids  = rating_counts[rating_counts.index.isin(['G', 'TV-G', 'TV-Y', 'TV-Y7', 'PG'])].sum()

print(f"Adult-rated (TV-MA + R)  : {adult:,}  ({adult/len(df)*100:.1f}%)")
print(f"Family/kids-friendly     : {kids:,}  ({kids/len(df)*100:.1f}%)")


## 6. Summary

**Key findings:**

| Question | Finding |
|---|---|
| Content split | 70% Movies, 30% TV Shows |
| Top country | United States — ~37% of all content |
| #2 country | India (Bollywood output is large) |
| Content peak | 2021 — highest number of additions |
| Top movie genre | Dramas |
| Top TV genre | International TV Shows |
| Most common rating | TV-MA — adult content dominates |

**Limitations:**
- `listed_in` contains multiple genres per title — only the first was used here
- `country` field also has multiple values for co-productions — only first country extracted
- Dataset has a cutoff date and does not include the most recent titles

**Possible extensions:**
- Explode the genre column and redo counts with all genres
- Time series analysis of genre trends by year
- Content-based recommendation system using genre + country + rating
